# Completed Matches

Query the gold `completed_matches` table — one row per finished fixture with
scores, league/season context, team names, and (where Matchbook odds data is
available) the pre-match favourite.

**Pre-requisite:** run the Dagster `espn_ingestion` job (which also runs the
dbt silver + gold models) so the data is populated.

In [ ]:
import os
import duckdb
import pandas as pd

DATA_DIR = os.environ.get("DATA_DIR", "/app/data")
PARQUET = f"{DATA_DIR}/gold/completed_matches.parquet"

In [ ]:
# All completed matches (most recent first)
df = duckdb.query(f"select * from read_parquet('{PARQUET}') order by kickoff_time desc").df()
print(f"{len(df):,} completed matches")
df.head(20)

In [ ]:
# Matches per league
duckdb.query(f"""
    select league, count(*) as matches
    from read_parquet('{PARQUET}')
    group by league
    order by matches desc
""").df()

In [ ]:
# Filter to a specific league — change 'eng.1' as needed
LEAGUE = "eng.1"

duckdb.query(f"""
    select kickoff_time, season, home_team, away_team, ft_score, favourite_team
    from read_parquet('{PARQUET}')
    where league = '{LEAGUE}'
    order by kickoff_time desc
""").df()

In [ ]:
# Matches where we have a Matchbook favourite (odds data linked)
duckdb.query(f"""
    select kickoff_time, league, home_team, away_team, ft_score, favourite_team
    from read_parquet('{PARQUET}')
    where favourite_team is not null
    order by kickoff_time desc
""").df()